# Klasifikasi Emosi Multimodal dengan Late Fusion
### Big Data Challenge (BDC) 2025

Notebook ini membangun model klasifikasi emosi (8 kelas: *Proud, Trust, Joy, Surprise, Neutral, Sadness, Fear, Anger*)
dari tiga modalitas data video, yaitu:

| Modalitas | Sumber Fitur |
|---|---|
| **Visual** | statistik frame video (brightness, motion, colorfulness, dll.) |
| **Audio** | fitur MFCC, chroma, mel-spectrogram, formant |
| **Teks** | embedding BERT dari transkrip ucapan |

**Strategi pemodelan — Late Fusion:**
1. Latih satu model klasifikasi terpisah untuk **setiap modalitas** (visual, audio, teks).
2. Ambil **probabilitas prediksi out-of-fold** dari ketiga model tersebut sebagai *meta-features*.
3. Latih **model fusi (meta-model)** di atas gabungan probabilitas tersebut untuk menghasilkan prediksi akhir.

Sebagai pembanding, disertakan juga eksperimen **Early Fusion** (menggabungkan seluruh fitur mentah menjadi satu
matriks lalu melatih satu model saja) beserta perbandingan tiga algoritma (XGBoost, LightGBM, Random Forest).

> **Catatan:** notebook ini berjalan di Google Colab dan membaca data dari Google Drive (folder `BDC_2025`).


## 1. Setup & Import Library

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import uniform, randint

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, ADASYN

warnings.filterwarnings("ignore")

### Mount Google Drive
Seluruh file data (fitur visual/audio/teks, label, dan template submission) disimpan di Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Konfigurasi

Semua konstanta (path data, mapping label, parameter cross-validation) dikumpulkan di satu tempat agar mudah
diubah tanpa perlu menelusuri seluruh notebook.


In [ ]:
# --- Path data (Google Drive) ---
BASE_DIR = "/content/drive/MyDrive/BDC_2025"

PATHS = {
    "visual_train": f"{BASE_DIR}/visual/eda_stats_train.csv",
    "visual_test":  f"{BASE_DIR}/visual/eda_stats_test.csv",
    "audio_train":  f"{BASE_DIR}/audio/temp_data_audio_advanced_feature_train.csv",
    "audio_test":   f"{BASE_DIR}/audio/temp_data_audio_advanced_feature_test_revisi.csv",
    "text_train":   f"{BASE_DIR}/teks/temp_data_bert_flattened_train.csv",
    "text_test":    f"{BASE_DIR}/teks/temp_data_bert_flattened_test_revisi.csv",
    "labels_train": f"{BASE_DIR}/data_train.csv",       # berisi kolom id, video, emotion
    "submission_template": f"{BASE_DIR}/submission.csv",
}

OUTPUT_DIR = "/content/drive/MyDrive/BDC_2025/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Mapping label ---
VALID_LABELS = [
    "Proud", "Trust", "Joy", "Surprise", "Neutral",
    "Sadness", "Fear", "Anger",
]
LABEL_MAP = {label: idx for idx, label in enumerate(VALID_LABELS)}
INV_LABEL_MAP = {idx: label for label, idx in LABEL_MAP.items()}
N_CLASSES = len(VALID_LABELS)

# --- Fitur audio & teks terpilih (hasil feature selection sebelumnya) ---
AUDIO_FEATURE_COLUMNS = [
    "id", "mfcc_17_skew", "chroma_1_std", "mfcc_15_skew", "chroma_3_std",
    "mfcc_17_kurtosis", "mel_5_std", "mel_5_mean", "mfcc_15_kurtosis",
    "mfcc_0_delta_std", "formant_2_std", "mfcc_19_skew", "mfcc_0_std",
    "mfcc_14_skew", "chroma_1_mean", "chroma_3_mean",
]
TEXT_FEATURE_COLUMNS = [
    "id", "424", "237", "133", "17", "164", "532", "667", "514", "68",
    "221", "240", "67", "613", "581", "232",
]

# --- Parameter eksperimen ---
N_FOLDS = 5
RANDOM_STATE = 42
VAL_SIZE = 0.15
BALANCING_METHOD = "adasyn"   # opsi: "smote" | "adasyn" | "class_weight" | None

## 3. Fungsi Bantu (Utility Functions)

Fungsi-fungsi berikut dipakai berulang kali di seluruh notebook (pembersihan label, balancing data,
serta pelatihan model dengan hyperparameter tuning), sehingga tidak perlu ditulis ulang di setiap sel.


In [ ]:
def clean_labels(df: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    """Buang baris dengan label di luar VALID_LABELS, lalu ubah label ke kode numerik."""
    df = df[df[label_col].isin(VALID_LABELS)].copy()
    df[label_col] = df[label_col].map(LABEL_MAP)
    return df


def balance_training_data(X_train, y_train, method: str = BALANCING_METHOD):
    """
    Menyeimbangkan data training yang tidak seimbang antar kelas.

    Mengembalikan (X_resampled, y_resampled, fit_params), di mana `fit_params`
    berisi `sample_weight` jika method == "class_weight" (kosong untuk method lain,
    karena SMOTE/ADASYN sudah menyeimbangkan datanya secara langsung).
    """
    fit_params = {}

    if method == "smote":
        min_count = pd.Series(y_train).value_counts().min()
        k_neighbors = max(1, min_count - 1)
        sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)
        X_res, y_res = sampler.fit_resample(X_train, y_train)

    elif method == "adasyn":
        min_count = pd.Series(y_train).value_counts().min()
        n_neighbors = max(1, min_count - 1)
        sampler = ADASYN(random_state=RANDOM_STATE, n_neighbors=n_neighbors)
        X_res, y_res = sampler.fit_resample(X_train, y_train)

    elif method == "class_weight":
        all_classes = np.arange(N_CLASSES)
        weights = compute_class_weight(class_weight="balanced", classes=all_classes, y=y_train)
        weight_map = dict(zip(all_classes, weights))
        fit_params["sample_weight"] = np.array([weight_map[label] for label in y_train])
        X_res, y_res = X_train, y_train

    else:
        X_res, y_res = X_train, y_train

    return X_res, y_res, fit_params


def train_with_tuning(model_name, X_train, y_train, fit_params=None, n_iter=20, cv=5):
    """
    Melatih model dengan RandomizedSearchCV, dioptimalkan terhadap macro F1-score.

    model_name : "xgboost" | "lightgbm" | "random_forest"
    Mengembalikan estimator terbaik (sudah di-refit pada seluruh X_train, y_train).
    """
    fit_params = fit_params or {}
    kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)

    if model_name == "xgboost":
        base_params = {"use_label_encoder": False, "eval_metric": "mlogloss", "seed": RANDOM_STATE}
        estimator = xgb.XGBClassifier(**base_params, tree_method="hist", device="cuda")
        param_dist = {
            "n_estimators": randint(100, 300),
            "max_depth": [2, 3, 4, 5, 6, 7],
            "learning_rate": [0.01, 0.02, 0.05, 0.1],
            "subsample": uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "gamma": [0, 0.1, 0.5, 1],
            "reg_alpha": [0.1, 1, 5, 10],
            "reg_lambda": [1.5, 2, 5, 10],
        }

    elif model_name == "lightgbm":
        base_params = {
            "objective": "multiclass", "metric": "multi_logloss",
            "random_state": RANDOM_STATE, "n_jobs": -1, "verbose": -1,
        }
        estimator = lgb.LGBMClassifier(**base_params, device="gpu")
        param_dist = {
            "n_estimators": randint(100, 300),
            "max_depth": [3, 5, 7],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "bagging_fraction": uniform(0.6, 0.4),
            "feature_fraction": uniform(0.6, 0.4),
            "num_leaves": randint(10, 50),
            "reg_alpha": [0.1, 1, 5, 10],
            "reg_lambda": [1.5, 2, 5, 10],
        }

    elif model_name == "random_forest":
        estimator = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
        param_dist = {
            "n_estimators": randint(100, 300),
            "max_depth": [5, 10, 20, 30, None],
            "max_features": ["sqrt", "log2", None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "bootstrap": [True, False],
            "criterion": ["gini", "entropy"],
        }

    else:
        raise ValueError(f"model_name tidak dikenal: {model_name}")

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring="f1_macro",
        cv=kfold,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_train, y_train, **fit_params)

    print(f"[{model_name}] Best macro F1 (CV): {search.best_score_:.4f}")
    print(f"[{model_name}] Best params: {search.best_params_}")
    return search.best_estimator_


def evaluate(model, X_val, y_val, label=""):
    """Cetak macro F1-score dan classification report pada set validasi."""
    preds = model.predict(X_val)
    f1_macro = f1_score(y_val, preds, average="macro")
    print(f"\n>>> {label} — Macro F1-Score (Validasi): {f1_macro:.4f} <<<")
    print(classification_report(y_val, preds, zero_division=0))
    return f1_macro

## 4. Memuat & Membersihkan Data per Modalitas

### 4.1 Label

In [ ]:
labels_raw = pd.read_csv(PATHS["labels_train"])
labels_raw = labels_raw.rename(columns={"emotion": "label"})
labels_raw = clean_labels(labels_raw, "label")
labels_raw[["id", "label"]].head()

### 4.2 Fitur Visual

Statistik ringkas dari setiap frame video (brightness, motion, colorfulness, edge density, dst.).

In [ ]:
visual_train = pd.read_csv(PATHS["visual_train"])
visual_train = clean_labels(visual_train, "label")

visual_test = pd.read_csv(PATHS["visual_test"])

VISUAL_FEATURE_COLS = [
    "brightness", "motion", "colorfulness",
    "edge_density", "optical_flow", "temporal_dynamics", "nframes",
]

print(f"Visual train: {visual_train.shape}, Visual test: {visual_test.shape}")
visual_train.head()

### 4.3 Fitur Audio

Fitur MFCC, chroma, mel-spectrogram, dan formant hasil ekstraksi audio, disaring ke 15 fitur terbaik.

In [ ]:
audio_train = pd.read_csv(PATHS["audio_train"])[AUDIO_FEATURE_COLUMNS]
audio_test = pd.read_csv(PATHS["audio_test"])[AUDIO_FEATURE_COLUMNS]

# Samakan nama kolom fitur (mfcc_1 ... mfcc_15) agar konsisten antara train & test
rename_map_train = {old: f"mfcc_{i+1}" for i, old in enumerate(audio_train.columns[1:])}
rename_map_test = {old: f"mfcc_{i+1}" for i, old in enumerate(audio_test.columns[1:])}
audio_train = audio_train.rename(columns=rename_map_train)
audio_test = audio_test.rename(columns=rename_map_test)

print(f"Audio train: {audio_train.shape}, Audio test: {audio_test.shape}")
audio_train.head()

### 4.4 Fitur Teks (BERT)

Embedding BERT dari transkrip ucapan, disaring ke 15 dimensi paling informatif.

In [ ]:
text_train = pd.read_csv(PATHS["text_train"])[TEXT_FEATURE_COLUMNS]
text_test_raw = pd.read_csv(PATHS["text_test"])

# Kolom test memiliki 'Unnamed: 0' tambahan dari proses ekspor sebelumnya
text_test = text_test_raw[TEXT_FEATURE_COLUMNS]

print(f"Teks train: {text_train.shape}, Teks test: {text_test.shape}")
text_train.head()

## 5. Menyatukan Data Antar Modalitas

Ketiga modalitas berasal dari file terpisah, sehingga perlu disatukan berdasarkan `id` video yang sama
agar setiap baris merepresentasikan video yang identik di seluruh modalitas.


In [ ]:
# id yang tersedia di ketiga modalitas sekaligus (irisan/intersection)
common_ids_train = (
    set(visual_train["id"])
    & set(audio_train["id"])
    & set(text_train["id"])
    & set(labels_raw["id"])
)
common_ids_train = sorted(common_ids_train)

print(f"Jumlah video (train) yang lengkap di 3 modalitas: {len(common_ids_train)}")

def align_by_id(df, ids, feature_cols):
    df = df[df["id"].isin(ids)].sort_values("id").reset_index(drop=True)
    return df[feature_cols].reset_index(drop=True)

X_visual = align_by_id(visual_train, common_ids_train, VISUAL_FEATURE_COLS)
X_audio = align_by_id(audio_train, common_ids_train, [c for c in audio_train.columns if c != "id"])
X_text = align_by_id(text_train, common_ids_train, [c for c in text_train.columns if c != "id"])

y_full = (
    labels_raw[labels_raw["id"].isin(common_ids_train)]
    .sort_values("id")["label"]
    .reset_index(drop=True)
    .values
)

print(f"X_visual: {X_visual.shape} | X_audio: {X_audio.shape} | X_text: {X_text.shape} | y: {y_full.shape}")

In [ ]:
# Data test: gunakan seluruh baris, urutkan berdasarkan id agar konsisten antar modalitas
test_ids = sorted(set(visual_test["id"]) & set(audio_test["id"]))

X_visual_test = align_by_id(visual_test, test_ids, VISUAL_FEATURE_COLS)
X_audio_test = align_by_id(audio_test, test_ids, [c for c in audio_test.columns if c != "id"])
X_text_test = text_test.reset_index(drop=True)  # sudah sejajar dengan urutan test_ids di sumbernya

print(f"X_visual_test: {X_visual_test.shape} | X_audio_test: {X_audio_test.shape} | X_text_test: {X_text_test.shape}")

### 5.1 Split Train / Validation

Pembagian data menjadi set training dan validasi dilakukan **sebelum balancing**, agar performa model
dievaluasi pada distribusi kelas yang natural (bukan hasil oversampling).


In [ ]:
idx_train, idx_val = train_test_split(
    np.arange(len(y_full)),
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

y_train, y_val = y_full[idx_train], y_full[idx_val]

print(f"Jumlah data training : {len(idx_train)}")
print(f"Jumlah data validasi : {len(idx_val)}")

## 6. Baseline — Early Fusion

Pendekatan paling sederhana: gabungkan seluruh fitur (visual + audio + teks) menjadi satu matriks besar,
lalu latih satu model klasifikasi di atasnya. Bagian ini juga dipakai untuk **membandingkan tiga algoritma**
(XGBoost, LightGBM, Random Forest) sebelum memutuskan pendekatan akhir.


In [ ]:
X_full = pd.concat([X_visual, X_audio, X_text], axis=1).values
X_full_test = pd.concat([X_visual_test, X_audio_test, X_text_test], axis=1).values

X_ef_train, X_ef_val = X_full[idx_train], X_full[idx_val]

X_ef_train_bal, y_ef_train_bal, ef_fit_params = balance_training_data(X_ef_train, y_train)
print(f"Shape sebelum balancing: {X_ef_train.shape} -> sesudah: {X_ef_train_bal.shape}")

### 6.1 XGBoost

In [ ]:
model_xgb = train_with_tuning("xgboost", X_ef_train_bal, y_ef_train_bal, ef_fit_params, n_iter=30)
f1_xgb = evaluate(model_xgb, X_ef_val, y_val, label="Early Fusion — XGBoost")

### 6.2 LightGBM

In [ ]:
model_lgbm = train_with_tuning("lightgbm", X_ef_train_bal, y_ef_train_bal, ef_fit_params, n_iter=20)
f1_lgbm = evaluate(model_lgbm, X_ef_val, y_val, label="Early Fusion — LightGBM")

### 6.3 Random Forest

In [ ]:
model_rf = train_with_tuning("random_forest", X_ef_train_bal, y_ef_train_bal, ef_fit_params, n_iter=15)
f1_rf = evaluate(model_rf, X_ef_val, y_val, label="Early Fusion — Random Forest")

### 6.4 Ringkasan Perbandingan Model (Early Fusion)

In [ ]:
comparison = pd.DataFrame({
    "model": ["XGBoost", "LightGBM", "Random Forest"],
    "macro_f1_validasi": [f1_xgb, f1_lgbm, f1_rf],
}).sort_values("macro_f1_validasi", ascending=False).reset_index(drop=True)

comparison

## 7. Late Fusion — Pendekatan Akhir

Pendekatan akhir yang dipakai untuk submission: melatih **satu model per modalitas** (visual, audio, teks),
lalu menggabungkan probabilitas prediksi ketiganya sebagai fitur untuk model fusi.

Agar model fusi tidak "curang" (melihat prediksi dari data yang sama yang dipakai model dasar untuk belajar),
probabilitas pada data training dihasilkan lewat **out-of-fold prediction**: setiap fold divalidasi oleh model
yang tidak dilatih pada fold tersebut.


In [ ]:
def get_oof_and_test_proba(X, y, X_test, model_name, n_folds=N_FOLDS, n_iter=15):
    """
    Menghasilkan probabilitas out-of-fold (OOF) untuk data training dan rata-rata
    probabilitas test dari model yang dilatih pada setiap fold.

    Mengembalikan (oof_proba, test_proba) dengan shape (n_samples, N_CLASSES).
    """
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)

    oof_proba = np.zeros((len(y), N_CLASSES))
    test_proba_folds = np.zeros((n_folds, len(X_test), N_CLASSES))

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y)):
        X_tr, X_va = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx], \
                     X.iloc[va_idx] if hasattr(X, "iloc") else X[va_idx]
        y_tr = y[tr_idx]

        X_tr_bal, y_tr_bal, fit_params = balance_training_data(X_tr, y_tr)

        model = train_with_tuning(model_name, X_tr_bal, y_tr_bal, fit_params, n_iter=n_iter, cv=3)

        oof_proba[va_idx] = model.predict_proba(X_va)
        test_proba_folds[fold] = model.predict_proba(X_test)

        print(f"  Fold {fold + 1}/{n_folds} selesai.")

    test_proba = test_proba_folds.mean(axis=0)
    return oof_proba, test_proba

### 7.1 Model Dasar per Modalitas

Setiap modalitas dilatih dengan model XGBoost (bisa diganti `lightgbm` / `random_forest` di parameter `model_name`).

In [ ]:
print("Melatih model dasar — Visual...")
oof_visual, test_proba_visual = get_oof_and_test_proba(
    X_visual, y_full, X_visual_test, model_name="xgboost"
)

print("\nMelatih model dasar — Audio...")
oof_audio, test_proba_audio = get_oof_and_test_proba(
    X_audio, y_full, X_audio_test, model_name="xgboost"
)

print("\nMelatih model dasar — Teks...")
oof_text, test_proba_text = get_oof_and_test_proba(
    X_text, y_full, X_text_test, model_name="xgboost"
)

### 7.2 Membentuk Meta-Features

In [ ]:
meta_features_train = np.concatenate([oof_visual, oof_audio, oof_text], axis=1)
meta_features_test = np.concatenate([test_proba_visual, test_proba_audio, test_proba_text], axis=1)

print(f"Meta-features train: {meta_features_train.shape}")  # (n_samples, N_CLASSES * 3)
print(f"Meta-features test : {meta_features_test.shape}")

### 7.3 Melatih & Mengevaluasi Model Fusi (Meta-Model)

Meta-features juga dibagi menjadi train/validasi (memakai split indeks yang sama seperti sebelumnya)
agar performa model fusi bisa dibandingkan secara adil dengan baseline early fusion di atas.


In [ ]:
X_meta_train, X_meta_val = meta_features_train[idx_train], meta_features_train[idx_val]

meta_model = xgb.XGBClassifier(
    use_label_encoder=False, eval_metric="mlogloss", seed=RANDOM_STATE,
    n_estimators=100, learning_rate=0.05,
)
meta_model.fit(X_meta_train, y_train)

f1_late_fusion = evaluate(meta_model, X_meta_val, y_val, label="Late Fusion — Meta-Model")

Dibandingkan dengan skor macro F1 baseline early fusion di atas, hasil ini menentukan apakah pendekatan
late fusion benar-benar memberi peningkatan performa. Jika ya, `meta_model` inilah yang dipakai untuk
prediksi final di bawah.


## 8. Melatih Ulang pada Seluruh Data & Prediksi Final

Setelah arsitektur & hyperparameter final ditentukan lewat validasi di atas, model fusi dilatih ulang
menggunakan **seluruh data training** (train + validasi) agar memanfaatkan data sebanyak mungkin sebelum
memprediksi data test.


In [ ]:
final_meta_model = xgb.XGBClassifier(
    use_label_encoder=False, eval_metric="mlogloss", seed=RANDOM_STATE,
    n_estimators=100, learning_rate=0.05,
)
final_meta_model.fit(meta_features_train, y_full)

final_predictions = final_meta_model.predict(meta_features_test)
final_predictions_label = [INV_LABEL_MAP[p] for p in final_predictions]

print(f"Jumlah prediksi test: {len(final_predictions)}")
print(f"10 prediksi pertama : {final_predictions_label[:10]}")

### 8.1 Menyimpan File Submission

In [ ]:
def save_submission_file(test_ids, predictions, output_dir, filename="submission.csv"):
    """Membuat dan menyimpan file submission.csv sesuai format yang diminta kompetisi."""
    if len(test_ids) != len(predictions):
        raise ValueError(
            f"Jumlah ID ({len(test_ids)}) tidak cocok dengan jumlah prediksi ({len(predictions)})!"
        )

    submission_df = pd.DataFrame({"id": test_ids, "predicted": predictions})
    submission_path = os.path.join(output_dir, filename)
    submission_df.to_csv(submission_path, index=False)

    print(f"File submission berhasil disimpan di: {submission_path}")
    print(submission_df.head())
    return submission_df


submission_df = save_submission_file(
    test_ids=test_ids,
    predictions=final_predictions,
    output_dir=OUTPUT_DIR,
    filename="submission_late_fusion.csv",
)

## 9. Kesimpulan

- Model **late fusion (stacking)** menggabungkan kekuatan tiap modalitas (visual, audio, teks) lewat
  probabilitas out-of-fold, sehingga meta-model belajar dari sinyal ketiga modalitas sekaligus tanpa risiko
  *data leakage*.
- Perbandingan dengan baseline **early fusion** (bagian 6) menunjukkan apakah pemisahan modalitas ini
  benar-benar bermanfaat dibanding menggabungkan semua fitur mentah sejak awal.
- File `submission_late_fusion.csv` berisi prediksi akhir untuk seluruh data test, siap diserahkan.
